# COGS515 Lab: Solving the 8-Puzzle with Search

**Week 2 · about 30 minutes · groups of 2–3**

**How this lab works**
- ✏️ means answer on the worksheet (pen and paper). 💻 means run a cell.
- **Predict first, then run.** Write your prediction on the worksheet *before* the cell is run.
- **No laptop?** No problem. The instructor runs this notebook on the projector, and every task can be answered on the worksheet.
- **With a laptop:** open this notebook in Google Colab (*File → Upload notebook*) or Jupyter. Nothing to install. Run the cells in order, but wait for the class at each ✏️ task.

| Part | Minutes |
|---|---|
| 1. Solve it yourself | 4 |
| 2. The problem formulation | 5 |
| 3. Breadth-first search | 8 |
| 4. Depth-first search | 6 |
| 5. Wrap-up | 3 |

## 1. Solve it yourself first ✏️ (4 min)

Slide tiles into the blank `_` until you reach the goal. Sliding a tile into the blank is the same as moving the blank the other way, so we describe each move by where the **blank** goes: up, down, left or right.

```
  Start (puzzle A)       Goal
      1 2 5              _ 1 2
      3 4 8              3 4 5
      _ 6 7              6 7 8
```

- ✏️ **Task 1a.** Solve it on paper. How many moves did you need?
- ✏️ **Task 1b.** Is your solution the shortest possible? How sure are you?
- ✏️ **Task 1c.** How did you decide which move to try next? Write down one rule of thumb you used.

*Don't run anything yet.*

## 2. The problem formulation (6 min)

Recall the components of a search problem from the lecture: **initial state**, **actions**, **transition model** (`result`), **goal test** and **path cost** (the sum of the action costs; every move costs 1 here).

**State.** A tuple of 9 numbers, read row by row, with `0` for the blank. The positions are numbered

```
0 1 2
3 4 5
6 7 8
```

so the goal `_ 1 2 / 3 4 5 / 6 7 8` is the tuple `(0, 1, 2, 3, 4, 5, 6, 7, 8)`.

**Action.** The position the blank moves to. For example, when the blank is at position 3 it can move to 0 (up), 4 (right) or 6 (down), so `actions` returns `(0, 4, 6)`. These are the lecture's *Up, Down, Left, Right*, written as the blank's new position so that `result` is a single swap.

**Result.** The blank and the tile at the chosen position swap places.

- ✏️ **Task 2a.** Write puzzle A as a tuple.
- ✏️ **Task 2b.** Predict `actions(A)`: which positions can the blank move to?
- ✏️ **Task 2c.** Predict `result(A, 7)`: draw the board after the blank moves to position 7.

Then run the three cells below to check 2a–2c.

In [ ]:
from collections import deque


class EightPuzzle:
    # Positions the blank can move to, for each position of the blank
    MOVES = ((1, 3), (0, 2, 4), (1, 5),
             (0, 4, 6), (1, 3, 5, 7), (2, 4, 8),
             (3, 7), (4, 6, 8), (5, 7))

    def __init__(self, initial, goal=(0, 1, 2, 3, 4, 5, 6, 7, 8)):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        return self.MOVES[state.index(0)]

    def result(self, state, action):
        s = list(state)
        blank = state.index(0)
        s[blank], s[action] = s[action], 0
        return tuple(s)

    def goal_test(self, state):
        return state == self.goal

    def action_cost(self, state1, action, state2):
        return 1


def show(*states):
    """Print one or more states side by side, with _ for the blank."""
    for row in range(3):
        print("   ".join(" ".join(str(t) if t else "_" for t in s[3 * row: 3 * row + 3]) for s in states))
    print()

In [ ]:
A = (1, 2, 5, 3, 4, 8, 0, 6, 7)    # Task 2a: is this your tuple?
puzzle = EightPuzzle(A)
show(A)

In [ ]:
print(puzzle.actions(A))           # Task 2b
show(puzzle.result(A, 7))          # Task 2c

## 3. Breadth-first search (7 min)

`Node` holds what the lecture's node slide lists (state, parent, action, path cost) plus its depth; `expand` creates a node's children. `bfs` is graph search with a FIFO queue:
- the frontier is a `deque`, and `popleft()` takes the **oldest** node;
- `reached` holds every state ever added to the frontier, so no state is added twice; it is the lecture's frontier and explored set in one, so one membership test replaces two;
- the goal test happens when a node is **removed** from the frontier.

`bfs` prints the length of the solution, the number of nodes expanded (removed from the frontier) and the largest size the frontier reached. Run the next two cells.

In [ ]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = parent.depth + 1 if parent else 0


def expand(problem, node):
    s = node.state
    for action in problem.actions(s):
        child_state = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, child_state)
        yield Node(child_state, node, action, cost)


def path(node):
    """The states from the initial state to node."""
    states = []
    while node is not None:
        states.append(node.state)
        node = node.parent
    return states[::-1]


def report(node, expanded, max_frontier):
    if node is None:
        print(f"No solution | nodes expanded: {expanded:,}")
    else:
        print(f"Solution: {node.depth:,} moves | nodes expanded: {expanded:,} | largest frontier: {max_frontier:,}")


def bfs(problem):
    frontier = deque([Node(problem.initial)])
    reached = {problem.initial}
    expanded, max_frontier = 0, 1
    while frontier:
        node = frontier.popleft()          # FIFO: oldest node first
        expanded += 1
        if problem.goal_test(node.state):
            report(node, expanded, max_frontier)
            return node
        for child in expand(problem, node):
            if child.state not in reached:
                reached.add(child.state)
                frontier.append(child)
        max_frontier = max(max_frontier, len(frontier))
    report(None, expanded, max_frontier)
    return None

In [ ]:
goal_node = bfs(EightPuzzle(A))
show(*path(goal_node))

- ✏️ **Task 3a.** Compare BFS's solution with yours from Task 1. Is the solution BFS finds always the shortest? Why?
- ✏️ **Task 3b.** Puzzles B and C need longer solutions. **Before** running each cell below, predict how many nodes BFS will expand. Then fill in the table on the worksheet.

```
  Puzzle B        Puzzle C
   5 6 2           7 2 4
   1 4 3           5 _ 6
   _ 7 8           8 3 1
```

In [ ]:
B = (5, 6, 2, 1, 4, 3, 0, 7, 8)
bfs(EightPuzzle(B));

In [ ]:
C = (7, 2, 4, 5, 0, 6, 8, 3, 1)
bfs(EightPuzzle(C));

- ✏️ **Task 3c.** From A to C the solution got about 4 times longer (6 → 26 moves). How many times more nodes did BFS expand? What does this say about BFS's time complexity, $O(b^d)$?
- ✏️ **Task 3d.** How many different arrangements of the 9 tiles (counting the blank) can the board show? *Hint:* count the ways to place 9 tiles on 9 squares.
- ✏️ **Task 3e.** Puzzle D is puzzle A with tiles 1 and 2 swapped. Can it be solved? What will BFS do? Predict, then run the cell. Compare the number of nodes expanded with your answer to Task 3d.

In [ ]:
D = (2, 1, 5, 3, 4, 8, 0, 6, 7)
bfs(EightPuzzle(D));

## 4. Depth-first search (6 min)

DFS uses a LIFO stack instead of a FIFO queue: it always takes the **newest** node. A `deque` can act as either.

- ✏️💻 **Task 4a.** The `dfs` cell below is a copy of `bfs` with one line missing. Fill in the line marked `TASK 4a` so that it removes the **newest** node from the frontier. *Hint:* a `deque` has `popleft()` and `pop()`.
- ✏️ **Task 4b.** Before running: on puzzle A, will DFS find a shorter or a longer solution than BFS? Will it expand more or fewer nodes? Will its largest frontier be smaller or bigger?

In [ ]:
def dfs(problem):
    frontier = deque([Node(problem.initial)])
    reached = {problem.initial}
    expanded, max_frontier = 0, 1
    while frontier:
        node = None                        # TASK 4a: remove the newest node from the frontier
        if node is None:
            print("Fill in the line marked TASK 4a first.")
            return None
        expanded += 1
        if problem.goal_test(node.state):
            report(node, expanded, max_frontier)
            return node
        for child in expand(problem, node):
            if child.state not in reached:
                reached.add(child.state)
                frontier.append(child)
        max_frontier = max(max_frontier, len(frontier))
    report(None, expanded, max_frontier)
    return None

In [ ]:
dfs(EightPuzzle(A));

Now run DFS on B and C too, and add the solution lengths to the table.

In [ ]:
dfs(EightPuzzle(B));
dfs(EightPuzzle(C));


- ✏️ **Task 4c.** Is DFS optimal? With a reached set, is it complete on the 8-puzzle?
- ✏️ **Task 4d.** The lecture said DFS needs only $O(bm)$ memory. Why did its frontier get so much bigger than BFS's on puzzle A? *Hint:* what is $m$ here?

## 5. Wrap-up (3 min)

You solved puzzle A in a few moves while keeping only a few boards in mind. BFS expanded 83 nodes to do the same, and DFS wandered for tens of thousands of moves. Neither algorithm uses any knowledge about the puzzle beyond its rules. That is what **uninformed** search means.

✏️ **Discussion** (in your group, then with the class)
1. Look at your rule of thumb from Task 1c. Could a program use it to decide which node to expand next?
2. BFS's frontier for puzzle C held about 25,000 nodes. Which parts of your own solving looked like BFS, which like DFS, and which like neither?

**Next:** rules of thumb that guide search are called *heuristics*; they are the topic of Week 3.